<p style="text-align: justify">Pelatihan jaringan dengan metode propagasi balik memerlukan proses iteratif yang dapat memakan waktu untuk menyesuaikan bobot dan bias. Karena itu, digunakan Extreme Learning Machine (ELM), yaitu metode pelatihan yang bersifat tidak iteratif. Bobot dan bias hanya disesuaikan melalui proses feed forward dengan satu kali pelatihan matriks.</p>
<img src="https://www.researchgate.net/publication/265608741/figure/fig3/AS:213795097059335@1427984131066/The-structure-of-extreme-learning-machine.png">
<p>Bobot antara lapisan input dan lapisan tersembunyi ditentukan secara acak. Bobot yang dilatih adalah bobot antara lapisan tersembunyi dan lapisan keluaran. Persamaannya adalah:</p>
<img src="https://wikimedia.org/api/rest_v1/media/math/render/svg/3c23b9696f0b9c900b68e71fc63b1b5a0e0cc1e1">
<ul><li><b>X</b>: matriks aktivasi lapisan tersembunyi</li><li><b>Xᵀ</b>: transpose dari <b>X</b></li><li><b>y</b>: label target</li></ul>

<b>Impor library</b>

In [1]:
import numpy as np
from pandas import read_csv

from sklearn.model_selection import train_test_split

<ul><li><i>NumPy</i> digunakan untuk operasi matriks.</li><li><i>Pandas</i> digunakan untuk memuat dataset.</li><li><i>train_test_split</i> digunakan untuk membagi data latih dan data uji.</li></ul>

<b>Muat dataset</b>

In [2]:
data = read_csv("seeds_data.csv")
data

,area,perimeter,compactness,length of kernel,width of kernel,asymmetry coefficient,length of kernel groove,class
0,15.2,14.8,0.870,5.76,3.31,2.22,5.20,1
1,14.8,14.5,0.881,5.55,3.33,1.01,4.95,1
2,14.2,14.0,0.900,5.29,3.33,2.69,4.82,1
3,13.8,13.9,0.895,5.32,3.37,2.25,4.80,1
4,16.1,14.9,0.903,5.65,3.56,1.35,5.17,1
...,...,...,...,...,...,...,...,...
205,12.1,13.0,0.878,5.13,2.98,3.63,4.80,3
206,11.2,12.8,0.851,5.10,2.79,4.32,5.00,3
207,13.0,13.6,0.888,5.23,3.23,8.31,5.05,3
208,11.8,13.2,0.852,5.17,2.83,3.59,5.04,3


<p>Dataset diperoleh dari <a href="https://archive.ics.uci.edu/ml/datasets/seeds">UCI Machine Learning Repository</a>. Nilai kosong pada dataset telah dibersihkan secara manual menggunakan metode <i>simple rolling mean</i>.</p>

<b>Tentukan data latih dan label klasifikasi</b>

In [3]:
x_train = data.drop(axis=1, columns='class')

labels = data['class']

Kolom selain <b>class</b> disimpan pada variabel <b>x_train</b>. Nilai pada kolom <b>class</b> disimpan pada variabel <b>labels</b>.

In [4]:
x_train.head()

,area,perimeter,compactness,length of kernel,width of kernel,asymmetry coefficient,length of kernel groove
0,15.2,14.8,0.870,5.76,3.31,2.22,5.20
1,14.8,14.5,0.881,5.55,3.33,1.01,4.95
2,14.2,14.0,0.900,5.29,3.33,2.69,4.82
3,13.8,13.9,0.895,5.32,3.37,2.25,4.80
4,16.1,14.9,0.903,5.65,3.56,1.35,5.17


In [5]:
labels.ravel()

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])

<b>Lakukan one-hot encoding</b>

In [6]:
kelas = len(np.unique(labels))

y_train = np.zeros([len(labels), kelas])

for i in range(len(labels)):
        if i < len(labels)/kelas:
            y_train[i][0] = 1
        elif i > len(labels)/kelas and i < len(labels)/kelas*2:
            y_train[i][1] = 1
        else:
            y_train[i][2] = 1

One-hot encoding mengubah setiap label menjadi representasi vektor biner. Format ini memudahkan perhitungan pada lapisan keluaran. Contohnya ditunjukkan di bawah ini.

In [7]:
print(y_train.shape)

y_train

(210, 3)


array([[1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [1., 0

Dataset memiliki 210 baris dan menghasilkan matriks target dengan tiga kolom. Setiap kolom mewakili satu kelas:<ul><li>Baris 1–70 mewakili kelas 1 dengan indeks <b>0</b> bernilai 1.</li><li>Baris 71–140 mewakili kelas 2 dengan indeks <b>1</b> bernilai 1.</li><li>Baris 141–210 mewakili kelas 3 dengan indeks <b>2</b> bernilai 1.</li></ul>

<b>Pisahkan data latih dan data uji</b>

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x_train, y_train, test_size=0.3)

print("Data latih :",len(x_train))
print("Data uji   :", len(x_test))

Data latih : 147
Data uji   : 63


Dataset dibagi menjadi 70% data latih dan 30% data uji.

<b>Inisialisasi fungsi perhitungan</b>

In [9]:
# Fungsi aktivasi
def sigmoid(x):
    return 1/(1 + np.exp(-x))

<img src="https://qph.fs.quoracdn.net/main-qimg-05edc1873d0103e36064862a45566dba"><p>Pada percobaan ini digunakan fungsi aktivasi sigmoid, yang menghasilkan nilai kontinu dalam rentang 0 sampai 1.</p>

In [10]:
def input_to_hidden(x):
    a = np.dot(x, Win)
    a += bias
    a = sigmoid(a)
    return a

Fungsi berikut mengalikan input dengan bobot dari lapisan input ke lapisan tersembunyi, kemudian menerapkan fungsi aktivasi.

In [11]:
def predict(x, bobot):
    x = input_to_hidden(x)
    y = np.dot(x, bobot)      # Perhitungan dari lapisan tersembunyi ke lapisan keluaran
    return y

Fungsi prediksi memanggil <b>input_to_hidden</b> untuk memproses data uji. Hasilnya dikalikan secara matriks dengan <b>bobot</b>, yang menyimpan bobot antara lapisan tersembunyi dan lapisan keluaran.

<b>Inisialisasi lapisan input dan lapisan tersembunyi</b>

In [12]:
inpt = x_train.shape[1]   # Jumlah node lapisan input
hidden = 100             # Jumlah node lapisan tersembunyi

bias = np.random.randn(hidden)
Win = np.random.normal(size=[inpt, hidden])

Win.shape

(7, 100)

<p>Jaringan terdiri atas 7 node pada lapisan input dan 100 node pada lapisan tersembunyi. Jumlah 7 node input sesuai dengan jumlah fitur yang digunakan untuk memprediksi kolom <b>class</b>.</p><p>Jumlah 100 node tersembunyi dipilih sebagai nilai percobaan; jumlah lain dapat diuji untuk mencari konfigurasi yang paling sesuai. Jumlah node lapisan keluaran ditentukan kemudian melalui perhitungan bobot dan cocok dengan jumlah label, yaitu tiga.</p>

<b>Perhitungan bobot ELM</b>

In [13]:
X = input_to_hidden(x_train)
Xt = np.transpose(X)

Wout = np.dot(np.linalg.pinv(np.dot(Xt, X)), np.dot(Xt, y_train))   # Bobot hasil pelatihan

Wout.shape

(100, 3)

<p>Fungsi <b>input_to_hidden</b> dipanggil dengan <b>x_train</b>, lalu hasilnya disimpan pada variabel <b>X</b>. Transpose dari <b>X</b> disimpan pada <b>Xt</b>.</p><p>Bobot antara lapisan tersembunyi dan lapisan keluaran dihitung menggunakan rumus berikut:</p><img src="https://wikimedia.org/api/rest_v1/media/math/render/svg/3c23b9696f0b9c900b68e71fc63b1b5a0e0cc1e1"><p>Dalam implementasi digunakan <b>np.linalg.pinv</b> sebagai pengganti <b>np.linalg.inv</b>. Matriks <b>XᵀX</b> dapat menjadi singular, sedangkan <b>pinv</b> menghitung invers semu melalui singular value decomposition. Hasilnya disimpan pada <b>Wout</b> dengan ukuran 100 × 3, sesuai dengan 100 node lapisan tersembunyi dan 3 node lapisan keluaran.</p>

<b>Evaluasi ELM</b>

Setelah seluruh bobot diperoleh, model dievaluasi menggunakan data uji.

In [15]:
y = predict(x_test, Wout)

betul = 0
total = len(y)

for i in range(total):
    hasil_prediksi = np.argmax(y[i])
    tes = np.argmax(y_test[i])
    betul = betul + (1 if hasil_prediksi == tes else 0)

print("Akurasi:", betul/total)

Akurasi: 0.9682539682539683


Akurasi model dapat dilihat pada keluaran sebelumnya. Bagian berikut membandingkan hasil prediksi dengan label sebenarnya.

In [16]:
np.argmax(y, axis=1)

array([1, 2, 0, 1, 1, 1, 1, 0, 2, 2, 2, 2, 0, 2, 2, 0, 1, 2, 1, 2, 0, 0,
       1, 0, 2, 1, 1, 2, 1, 0, 0, 0, 0, 2, 1, 0, 0, 1, 1, 0, 0, 2, 1, 0,
       0, 0, 1, 0, 0, 0, 0, 2, 1, 2, 2, 1, 1, 2, 0, 0, 0, 0, 0])

In [17]:
np.argmax(y_test, axis=1)

array([1, 2, 0, 1, 1, 1, 1, 0, 2, 2, 2, 2, 0, 2, 2, 0, 1, 2, 1, 2, 0, 0,
       1, 0, 2, 1, 1, 2, 1, 0, 0, 0, 0, 2, 1, 0, 0, 1, 1, 0, 0, 2, 1, 0,
       0, 0, 1, 0, 0, 2, 0, 0, 1, 2, 2, 1, 1, 2, 0, 0, 0, 0, 0])